# Mask2Former — 2D RGB Image Segmentation

**Model:** Mask2Former + ResNet-50 backbone (COCO Panoptic Segmentation)  
**Config:** `configs/coco/panoptic-segmentation/maskformer2_R50_bs16_50ep.yaml`  
**Weights:** Auto-downloaded (~200 MB) if not present locally.

**Outputs:**
- Panoptic segmentation (things + stuff unified)
- Instance segmentation (per-object masks + scores)
- Semantic segmentation (per-pixel class labels)

## 1. Imports

In [ ]:
import os
import sys
import urllib.request

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Make sure the repo root is on the path
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from detectron2.config import get_cfg
from detectron2.data import MetadataCatalog
from detectron2.data.detection_utils import read_image
from detectron2.engine import DefaultPredictor
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.utils.visualizer import ColorMode, Visualizer

from mask2former import add_maskformer2_config

print("Imports OK")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
# ── User settings ──────────────────────────────────────────────────────────────
INPUT_IMAGE   = "path/to/your/image.jpg"   # <-- change this
OUTPUT_IMAGE  = "segmentation_output.png"   # saved in current directory
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
CONFIDENCE_THRESHOLD = 0.5
# ───────────────────────────────────────────────────────────────────────────────

MODEL_URL     = (
    "https://dl.fbaipublicfiles.com/maskformer/mask2former/coco/panoptic/"
    "maskformer2_R50_bs16_50ep/model_final_94dc52.pkl"
)
WEIGHTS_PATH  = os.path.join(REPO_ROOT, "model_final_94dc52.pkl")
CONFIG_FILE   = os.path.join(
    REPO_ROOT,
    "configs/coco/panoptic-segmentation/maskformer2_R50_bs16_50ep.yaml"
)

print(f"Device      : {DEVICE}")
print(f"Config file : {CONFIG_FILE}")
print(f"Weights     : {WEIGHTS_PATH}")

## 3. Download Model Weights (if needed)

In [ ]:
if not os.path.isfile(WEIGHTS_PATH):
    print(f"Weights not found. Downloading from:\n  {MODEL_URL}")
    urllib.request.urlretrieve(MODEL_URL, WEIGHTS_PATH)
    print("Download complete.")
else:
    print(f"Weights already present: {WEIGHTS_PATH}")

## 4. Build Predictor

In [ ]:
cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
cfg.merge_from_file(CONFIG_FILE)

cfg.MODEL.WEIGHTS = WEIGHTS_PATH
cfg.MODEL.DEVICE  = DEVICE
cfg.MODEL.MASK_FORMER.TEST.SEMANTIC_ON  = True
cfg.MODEL.MASK_FORMER.TEST.INSTANCE_ON  = True
cfg.MODEL.MASK_FORMER.TEST.PANOPTIC_ON  = True
cfg.MODEL.RETINANET.SCORE_THRESH_TEST   = CONFIDENCE_THRESHOLD
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST   = CONFIDENCE_THRESHOLD
cfg.MODEL.PANOPTIC_FPN.COMBINE.INSTANCES_CONFIDENCE_THRESH = CONFIDENCE_THRESHOLD
cfg.freeze()

predictor = DefaultPredictor(cfg)
metadata  = MetadataCatalog.get("coco_2017_val_panoptic")

print("Predictor ready.")

## 5. Run Inference

In [ ]:
assert os.path.isfile(INPUT_IMAGE), f"Image not found: {INPUT_IMAGE}"

image = read_image(INPUT_IMAGE, format="BGR")   # (H, W, 3) BGR
print(f"Image shape: {image.shape}")

with torch.no_grad():
    predictions = predictor(image)

print("Keys in predictions:", list(predictions.keys()))
if "instances" in predictions:
    print(f"Detected instances: {len(predictions['instances'])}")

## 6. Visualize Results

In [ ]:
image_rgb = image[:, :, ::-1]   # BGR → RGB for matplotlib / Visualizer

# — Panoptic ——————————————————————————————————————————————————————————————————
v = Visualizer(image_rgb, metadata, scale=1.2, instance_mode=ColorMode.IMAGE_BW)
panoptic_seg, segments_info = predictions["panoptic_seg"]
panoptic_out = v.draw_panoptic_seg_predictions(
    panoptic_seg.to("cpu"), segments_info
).get_image()

# — Instance ——————————————————————————————————————————————————————————————————
v = Visualizer(image_rgb, metadata, scale=1.2, instance_mode=ColorMode.IMAGE_BW)
instance_out = v.draw_instance_predictions(
    predictions["instances"].to("cpu")
).get_image()

# — Semantic ——————————————————————————————————————————————————————————————————
v = Visualizer(image_rgb, metadata, scale=1.2, instance_mode=ColorMode.IMAGE_BW)
semantic_out = v.draw_sem_seg(
    predictions["sem_seg"].argmax(dim=0).to("cpu")
).get_image()

# — Display side-by-side ——————————————————————————————————————————————————————
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, img, title in zip(
    axes,
    [panoptic_out, instance_out, semantic_out],
    ["Panoptic Segmentation", "Instance Segmentation", "Semantic Segmentation"],
):
    ax.imshow(img)
    ax.set_title(title, fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Save Combined Output

In [ ]:
def add_label(img: np.ndarray, text: str) -> np.ndarray:
    out = img.copy()
    cv2.putText(out, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1.0, (255, 255, 255), 2, cv2.LINE_AA)
    return out

combined = np.concatenate([
    add_label(panoptic_out,  "Panoptic Segmentation"),
    add_label(instance_out,  "Instance Segmentation"),
    add_label(semantic_out,  "Semantic Segmentation"),
], axis=0)

cv2.imwrite(OUTPUT_IMAGE, combined[:, :, ::-1])   # RGB → BGR for OpenCV
print(f"Saved: {OUTPUT_IMAGE}")

## 8. Inspect Instance Predictions

In [ ]:
instances = predictions["instances"].to("cpu")
print(f"Total instances detected: {len(instances)}\n")

if instances.has("pred_classes") and hasattr(metadata, "thing_classes"):
    print(f"{'#':<4} {'Class':<22} {'Score':>6}")
    print("-" * 35)
    for i, (cls_id, score) in enumerate(
        zip(instances.pred_classes.tolist(), instances.scores.tolist())
    ):
        label = metadata.thing_classes[cls_id]
        print(f"{i:<4} {label:<22} {score:>6.3f}")